
# SU(3) $O(u^6)$ glueball rest mass — safe audit V5

This replaces the oversized V4 notebook. V4 embedded a 16 MB archive as a 21 MB base64 code cell, which can destabilize the Colab browser and runtime.

V5 is deliberately small and uses one companion bundle. It does not extract or load the 354 MiB ordered-word TSV into memory. The default run streams it directly from the ZIP, verifies its SHA-256 and row count, checks Stage 1 and every internal-release manifest entry, and exactly recomputes the dominant eight-block anchor.

**Scope:** this verifies the provisional internal certificate. It does **not** independently regenerate the five missing large intermediates or rerun the complete 205,699-block contraction. Therefore it does not claim that $m_6$ is independently solved.


In [ ]:

from pathlib import Path
import hashlib
import os
import shutil
import subprocess
import sys
import zipfile

BUNDLE_NAME_PREFIX = "SU3_Y6_M6_SAFE_AUDIT_V5_BUNDLE"
EXPECTED_BUNDLE_SHA256 = "65ec7a2f5e037cc9050155cad7e2317f1ba3782fc803b4ccf5dffc60d2c803a4"
FULL_STAGE0_STREAM_CHECK = True   # True: verify all 3,094,806 rows; False: faster manifest-only check

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def valid_candidates():
    roots = [Path("/content"), Path("/mnt/data"), Path.cwd()]
    seen = set()
    for root in roots:
        if not root.exists():
            continue
        for p in sorted(root.glob(BUNDLE_NAME_PREFIX + "*.zip")):
            rp = p.resolve()
            if rp in seen:
                continue
            seen.add(rp)
            try:
                if sha256_file(rp) == EXPECTED_BUNDLE_SHA256:
                    yield rp
            except OSError:
                pass

bundle = next(valid_candidates(), None)
if bundle is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            f"Place {BUNDLE_NAME_PREFIX}.zip beside this notebook. "
            f"Expected SHA-256: {EXPECTED_BUNDLE_SHA256}"
        ) from exc
    print("Upload SU3_Y6_M6_SAFE_AUDIT_V5_BUNDLE.zip")
    files.upload()
    bundle = next(valid_candidates(), None)

if bundle is None:
    raise FileNotFoundError(
        "No uploaded bundle matched the required SHA-256. "
        "Do not rename or substitute an older V3/V4 package."
    )

print("Bundle:", bundle)
print("Bundle SHA-256:", EXPECTED_BUNDLE_SHA256)

runtime_root = Path("/content") if Path("/content").exists() else Path.cwd()
extract_root = runtime_root / "SU3_Y6_M6_SAFE_AUDIT_V5_EXTRACTED"
marker = extract_root / ".bundle_sha256"
if not marker.is_file() or marker.read_text().strip() != EXPECTED_BUNDLE_SHA256:
    if extract_root.exists():
        shutil.rmtree(extract_root)
    extract_root.mkdir(parents=True)
    with zipfile.ZipFile(bundle) as zf:
        zf.extractall(extract_root)
    marker.write_text(EXPECTED_BUNDLE_SHA256 + "\n")

package = extract_root / "SU3_Y6_M6_SAFE_AUDIT_V5"
verifier = package / "verify_safe_audit_v5.py"
if not verifier.is_file():
    raise FileNotFoundError(f"Verifier missing after extraction: {verifier}")

report = runtime_root / "AUDIT_Y6_su3_m6_safe_v5_report.json"
cmd = [
    sys.executable,
    "-u",
    str(verifier),
    "--bundle-dir", str(package),
    "--report", str(report),
]
if not FULL_STAGE0_STREAM_CHECK:
    cmd.append("--fast")

print("Running low-memory audit...")
subprocess.run(cmd, check=True, env={**os.environ, "PYTHONUNBUFFERED": "1"})
print("\nReport written to:", report)



## Interpretation

A `PASS SAFE AUDIT` result establishes that the supplied Stage-0 corpus, Stage-1 local library, internal certificates, hashes, counts, folded identities, and dominant anchor are mutually consistent.

It does not regenerate these five absent contraction inputs:

- `Y6_CLASS_ENERGY_SPECTRA.bin`
- `Y6_ENERGY_CLASSES.tsv`
- `Y6_EXACT_LOCAL_PATH_TENSORS.json.gz`
- `Y6_GAMMA_TOPOLOGY_BLOCKS.tsv`
- `Y6_GLOBAL_FOLDED_WEIGHT_CATALOG.tsv`

Until those are independently rebuilt and the complete contraction is rerun, the displayed $m_6$ remains provisional.


In [ ]:
# Optional: download the compact JSON audit report from Colab.
# Uncomment the final two lines after the audit has passed.
report_path = report
print(report_path, "exists =", report_path.exists())
# from google.colab import files
# files.download(str(report_path))
